# NumPy Weak-Spot Fluency Drills

A follow-up drill ladder focused on the mistakes from the previous NumPy muscle-memory notebook: slicing, fancy indexing, `np.where` indices, axis reductions, standardization, padding, diagonals, broadcasting, pairwise distances, and `einsum`.


In [ ]:
import numpy as np

np.set_printoptions(precision=3, suppress=True)

# Shared arrays for the drills. Treat these as read-only.
v = np.array([6, -2, 0, 9, -5, 4, 11, -8])
scores = np.array([61, 88, 47, 93, 75, 52, 100, 69, 84, 38, 91, 73])
labels = np.array(["red", "blue", "red", "green", "blue", "red", "yellow", "green"])
class_ids = np.array([2, 0, 3, 1, 2, 1])

G = np.arange(1, 21).reshape(4, 5)
H = np.array([
    [0, 2, 0, 4, 5],
    [7, 0, 0, 1, 0],
    [3, 3, 0, 0, 9],
    [0, 6, 8, 0, 0],
])

Z = np.array([
    [2.0, 5.0, 8.0, 11.0],
    [4.0, 7.0, 10.0, 13.0],
    [6.0, 9.0, 12.0, 15.0],
    [8.0, 11.0, 14.0, 17.0],
    [10.0, 13.0, 16.0, 19.0],
])

points = np.array([
    [0.0, 0.0, 1.0],
    [1.0, 2.0, 2.0],
    [3.0, 1.0, 0.0],
    [4.0, 4.0, 4.0],
    [2.0, 3.0, 5.0],
])

W = np.array([
    [1.0, 0.5],
    [-1.0, 2.0],
    [0.25, -0.5],
])

image = np.arange(5 * 6 * 3).reshape(5, 6, 3)
walk = np.array([3, -1, -2, 5, -3, 4, -6, 2])


def _same(got, expected):
    if isinstance(expected, np.ndarray):
        if not isinstance(got, np.ndarray) or got.shape != expected.shape:
            return False
        if expected.dtype.kind in "OUS":
            return np.array_equal(got, expected)
        return np.allclose(got, expected)
    if isinstance(expected, tuple):
        return isinstance(got, tuple) and len(got) == len(expected) and all(_same(g, e) for g, e in zip(got, expected))
    if isinstance(expected, list):
        return isinstance(got, list) and len(got) == len(expected) and all(_same(g, e) for g, e in zip(got, expected))
    if isinstance(expected, (float, np.floating)):
        return np.allclose(got, expected)
    return got == expected


def check(challenge_id, answer):
    expected = EXPECTED[challenge_id]
    if answer is None:
        print(f"{challenge_id}: not solved yet")
        return
    ok = _same(answer, expected)
    if ok:
        print(f"{challenge_id}: correct")
    else:
        print(f"{challenge_id}: not yet")
        print("got:", answer)
        if isinstance(answer, np.ndarray):
            print("got shape:", answer.shape)
        print("expected shape:", expected.shape if isinstance(expected, np.ndarray) else type(expected).__name__)


# Reference answers for the checker. Do not inspect this block while drilling.
EXPECTED = {
    "c01": v[1::2],
    "c02": v[::-1],
    "c03": G[:, ::2],
    "c04": G[::-1],
    "c05": G[:, ::-1],
    "c06": G[1:4, 1::2],
    "c07": G[-2:, :3],
    "c08": G[::2, ::-2],
    "c09": np.where(scores >= 80)[0],
    "c10": np.where(v < 0)[0],
    "c11": np.where(scores < 60, 60, scores),
    "c12": v[[5, 0, 3, 3]],
    "c13": G[[3, 0, 2]],
    "c14": G[[0, 1, 3], [4, 0, 2]],
    "c15": G[np.ix_([0, 2, 3], [1, 4])],
    "c16": scores[np.argsort(scores)[-4:]],
    "c17": np.argsort(scores)[:3],
    "c18": labels[[6, 1, 0, 3]],
    "c19": np.count_nonzero(H, axis=1),
    "c20": H.sum(axis=0),
    "c21": G.max(axis=1),
    "c22": image.mean(axis=(0, 1)),
    "c23": image.max(axis=(0, 1)),
    "c24": np.count_nonzero(image > 40, axis=(0, 1)),
    "c25": Z.mean(axis=0, keepdims=True),
    "c26": np.linalg.norm(points, axis=1),
    "c27": points / np.linalg.norm(points, axis=1, keepdims=True),
    "c28": Z - Z.mean(axis=0, keepdims=True),
    "c29": (Z - Z.mean(axis=0, keepdims=True)) / Z.std(axis=0, keepdims=True),
    "c30": (Z - Z.min(axis=0, keepdims=True)) / (Z.max(axis=0, keepdims=True) - Z.min(axis=0, keepdims=True)),
    "c31": G + np.array([10, 20, 30, 40, 50]),
    "c32": Z - Z.mean(axis=1, keepdims=True),
    "c33": Z * np.array([1.0, 10.0, 100.0, 1000.0]),
    "c34": points - np.array([1.0, 1.0, 1.0]),
    "c35": np.pad(v, (1, 2), constant_values=-1),
    "c36": np.pad(G, ((1, 1), (2, 2)), constant_values=0),
    "c37": np.diag([2, 4, 6, 8]),
    "c38": np.diag(G[:, :4]),
    "c39": np.eye(5, dtype=int),
    "c40": np.sort(G, axis=1),
    "c41": np.sort(scores)[-5:],
    "c42": ((points[:, None, :] - points[None, :, :]) ** 2).sum(axis=2),
    "c43": np.abs(v[:, None] - v[None, :]),
    "c44": np.einsum("ij,j->i", points, np.array([0.2, 0.3, 0.5])),
    "c45": np.einsum("ij,jk->ik", points, W),
    "c46": image[1:4, 2:5, :],
    "c47": image - image.mean(axis=(0, 1), keepdims=True),
    "c48": image.reshape(-1, 3),
    "c49": np.eye(4, dtype=int)[class_ids],
    "c50": np.diff(np.cumsum(walk)),
}



## Drill Rules

- Do not edit the setup cell except to rerun it.
- For each challenge, replace `answer = None` with one NumPy expression.
- Before running a harder cell, write the expected output shape as a comment.
- If a cell is wrong, inspect the printed shape before inspecting values.


## Drill Ladder


In [ ]:
# Challenge 01 - step slicing vector
# Task: Select every other element of v, starting at index 1.
# Expected shape: write it here before solving.
answer = None

check("c01", answer)


In [ ]:
# Challenge 02 - reverse vector
# Task: Reverse v.
# Expected shape: write it here before solving.
answer = None

check("c02", answer)


In [ ]:
# Challenge 03 - step slicing columns
# Task: Select every other column of G, starting from column 0.
# Expected shape: write it here before solving.
answer = None

check("c03", answer)


In [ ]:
# Challenge 04 - reverse rows
# Task: Reverse the row order of G.
# Expected shape: write it here before solving.
answer = None

check("c04", answer)


In [ ]:
# Challenge 05 - reverse columns
# Task: Reverse the column order of G.
# Expected shape: write it here before solving.
answer = None

check("c05", answer)


In [ ]:
# Challenge 06 - mixed row and column slicing
# Task: Select rows 1:4 of G and columns 1, 3.
# Expected shape: write it here before solving.
answer = None

check("c06", answer)


In [ ]:
# Challenge 07 - negative row slicing
# Task: Select the last two rows and first three columns of G.
# Expected shape: write it here before solving.
answer = None

check("c07", answer)


In [ ]:
# Challenge 08 - combined step and reverse
# Task: Select every other row of G, and columns from right to left with step 2.
# Expected shape: write it here before solving.
answer = None

check("c08", answer)


In [ ]:
# Challenge 09 - where indices high scores
# Task: Return indices where scores are at least 80.
# Expected shape: write it here before solving.
answer = None

check("c09", answer)


In [ ]:
# Challenge 10 - where indices negatives
# Task: Return indices where v is negative.
# Expected shape: write it here before solving.
answer = None

check("c10", answer)


In [ ]:
# Challenge 11 - where values threshold
# Task: Replace scores below 60 with 60, preserving shape.
# Expected shape: write it here before solving.
answer = None

check("c11", answer)


In [ ]:
# Challenge 12 - fancy indexing vector
# Task: From v, select positions [5, 0, 3, 3] in that order.
# Expected shape: write it here before solving.
answer = None

check("c12", answer)


In [ ]:
# Challenge 13 - fancy indexing rows
# Task: From G, select rows [3, 0, 2] in that order.
# Expected shape: write it here before solving.
answer = None

check("c13", answer)


In [ ]:
# Challenge 14 - paired fancy indexing
# Task: From G, select values at coordinate pairs (0,4), (1,0), (3,2).
# Expected shape: write it here before solving.
answer = None

check("c14", answer)


In [ ]:
# Challenge 15 - rectangular fancy indexing
# Task: From G, select rows [0, 2, 3] crossed with columns [1, 4].
# Expected shape: write it here before solving.
answer = None

check("c15", answer)


In [ ]:
# Challenge 16 - argsort top values
# Task: Return the top 4 scores using argsort. Ascending order is fine.
# Expected shape: write it here before solving.
answer = None

check("c16", answer)


In [ ]:
# Challenge 17 - argsort bottom indices
# Task: Return the indices of the bottom 3 scores.
# Expected shape: write it here before solving.
answer = None

check("c17", answer)


In [ ]:
# Challenge 18 - fancy indexing labels
# Task: From labels, select positions [6, 1, 0, 3] in that order.
# Expected shape: write it here before solving.
answer = None

check("c18", answer)


In [ ]:
# Challenge 19 - count nonzero by row
# Task: Count nonzero entries in each row of H.
# Expected shape: write it here before solving.
answer = None

check("c19", answer)


In [ ]:
# Challenge 20 - sum by column
# Task: Compute the sum of each column in H.
# Expected shape: write it here before solving.
answer = None

check("c20", answer)


In [ ]:
# Challenge 21 - max by row
# Task: Compute the max value of each row in G.
# Expected shape: write it here before solving.
answer = None

check("c21", answer)


In [ ]:
# Challenge 22 - image mean per channel
# Task: Compute the mean value per image channel.
# Expected shape: write it here before solving.
answer = None

check("c22", answer)


In [ ]:
# Challenge 23 - image max per channel
# Task: Compute the max value per image channel.
# Expected shape: write it here before solving.
answer = None

check("c23", answer)


In [ ]:
# Challenge 24 - image count per channel
# Task: Count values greater than 40 separately for each image channel.
# Expected shape: write it here before solving.
answer = None

check("c24", answer)


In [ ]:
# Challenge 25 - keepdims column mean
# Task: Compute the mean of each column in Z, keeping the reduced axis.
# Expected shape: write it here before solving.
answer = None

check("c25", answer)


In [ ]:
# Challenge 26 - row l2 norms
# Task: Compute the L2 norm of each row in points.
# Expected shape: write it here before solving.
answer = None

check("c26", answer)


In [ ]:
# Challenge 27 - row normalization
# Task: Normalize each row of points to unit length.
# Expected shape: write it here before solving.
answer = None

check("c27", answer)


In [ ]:
# Challenge 28 - column centering
# Task: Column-center Z by subtracting each column mean.
# Expected shape: write it here before solving.
answer = None

check("c28", answer)


In [ ]:
# Challenge 29 - column standardization
# Task: Standardize each column of Z to mean 0 and standard deviation 1.
# Expected shape: write it here before solving.
answer = None

check("c29", answer)


In [ ]:
# Challenge 30 - min max scaling
# Task: Scale each column of Z to the range 0 to 1.
# Expected shape: write it here before solving.
answer = None

check("c30", answer)


In [ ]:
# Challenge 31 - broadcast row vector
# Task: Add [10, 20, 30, 40, 50] to every row of G.
# Expected shape: write it here before solving.
answer = None

check("c31", answer)


In [ ]:
# Challenge 32 - row centering
# Task: Subtract each row's own mean from that row in Z.
# Expected shape: write it here before solving.
answer = None

check("c32", answer)


In [ ]:
# Challenge 33 - broadcast column weights
# Task: Multiply Z's columns by [1, 10, 100, 1000].
# Expected shape: write it here before solving.
answer = None

check("c33", answer)


In [ ]:
# Challenge 34 - subtract center point
# Task: Subtract [1, 1, 1] from every row of points.
# Expected shape: write it here before solving.
answer = None

check("c34", answer)


In [ ]:
# Challenge 35 - asymmetric pad vector
# Task: Pad v with one -1 on the left and two -1 values on the right.
# Expected shape: write it here before solving.
answer = None

check("c35", answer)


In [ ]:
# Challenge 36 - pad matrix border
# Task: Pad G with one zero row above/below and two zero columns left/right.
# Expected shape: write it here before solving.
answer = None

check("c36", answer)


In [ ]:
# Challenge 37 - diagonal matrix
# Task: Create a diagonal matrix with diagonal [2, 4, 6, 8].
# Expected shape: write it here before solving.
answer = None

check("c37", answer)


In [ ]:
# Challenge 38 - extract diagonal
# Task: Extract the diagonal from the first four columns of G.
# Expected shape: write it here before solving.
answer = None

check("c38", answer)


In [ ]:
# Challenge 39 - integer identity
# Task: Create a 5 by 5 integer identity matrix.
# Expected shape: write it here before solving.
answer = None

check("c39", answer)


In [ ]:
# Challenge 40 - sort each row
# Task: Sort each row of G in ascending order.
# Expected shape: write it here before solving.
answer = None

check("c40", answer)


In [ ]:
# Challenge 41 - top sorted values
# Task: Return the top 5 scores in ascending order.
# Expected shape: write it here before solving.
answer = None

check("c41", answer)


In [ ]:
# Challenge 42 - pairwise squared distances
# Task: Compute the full pairwise squared-distance matrix between rows of points.
# Expected shape: write it here before solving.
answer = None

check("c42", answer)


In [ ]:
# Challenge 43 - pairwise absolute differences
# Task: Compute the full pairwise absolute-difference matrix for v.
# Expected shape: write it here before solving.
answer = None

check("c43", answer)


In [ ]:
# Challenge 44 - einsum weighted sum
# Task: Use np.einsum to compute each point's weighted sum with weights [0.2, 0.3, 0.5].
# Expected shape: write it here before solving.
answer = None

check("c44", answer)


In [ ]:
# Challenge 45 - einsum matrix product
# Task: Use np.einsum to compute points @ W.
# Expected shape: write it here before solving.
answer = None

check("c45", answer)


In [ ]:
# Challenge 46 - 3d crop
# Task: Crop image rows 1:4 and columns 2:5, keeping all channels.
# Expected shape: write it here before solving.
answer = None

check("c46", answer)


In [ ]:
# Challenge 47 - per channel centering
# Task: Subtract the per-channel mean from image, preserving image shape.
# Expected shape: write it here before solving.
answer = None

check("c47", answer)


In [ ]:
# Challenge 48 - flatten spatial dimensions
# Task: Reshape image into a 2D array with one row per pixel and three channel columns.
# Expected shape: write it here before solving.
answer = None

check("c48", answer)


In [ ]:
# Challenge 49 - one hot via fancy indexing
# Task: Create one-hot rows for class_ids using an integer identity matrix with 4 classes.
# Expected shape: write it here before solving.
answer = None

check("c49", answer)


In [ ]:
# Challenge 50 - diff after cumsum
# Task: Compute consecutive differences of the cumulative walk positions.
# Expected shape: write it here before solving.
answer = None

check("c50", answer)


## Stop Condition

You are done when:
- c01-c18 feel automatic without reading the old notebook notes.
- c19-c34 are solved by predicting the output shape first.
- c35-c50 can be debugged by printing only `.shape`, not full arrays.

Contest reflex: name the surviving axes before writing the NumPy expression.
